# Spatially join points and polygons from feature layers


Spatially join points and polygons from public hosted feature services and inspect the results in a notebook.

This version avoids the ArcGIS Python API. It uses the ArcGIS REST query endpoint only to retrieve public data, then uses GeoPandas/Shapely for spatial operations and Folium for interactive mapping. It does not publish results back to ArcGIS Online.

> If you want to use local files such as CSV or shapefiles instead, the data-loading section can be adapted to use `geopandas.read_file()` or `pandas.read_csv()`.

## 1. Import libraries

This version uses the ArcGIS REST API to download public feature-layer data, then uses GeoPandas/Shapely for spatial analysis and Folium for mapping. It does **not** use the ArcGIS Python API or require an ArcGIS Online sign-in.

In [1]:
import requests
import pandas as pd
import geopandas as gpd
import folium
from IPython.display import display

In [2]:
# No GIS login is needed for public feature services.
# If a layer is private, you will need to provide an appropriate token.

## 2. Define inputs


In [3]:
# Public ArcGIS Feature Service layer URLs
polygon_layer_url = "https://services1.arcgis.com/KNdRU5cN6ENqCTjk/arcgis/rest/services/CSU_ELC_Boundary/FeatureServer/0"
point_layer_url = "https://services2.arcgis.com/PYn6bWCjT6bhw1z3/ArcGIS/rest/services/ColoradoPoints/FeatureServer/0"

# Coordinate system used when downloading data (WGS 84)
coordinate_system_id = 4326  # EPSG:4326

## 3. Fetch the point and polygon layers


The helper below queries a public ArcGIS Feature Service through its REST endpoint. It requests GeoJSON in WGS 84 (EPSG:4326) and downloads records in batches so it can handle layers larger than a single response. The service must allow public queries.

In [4]:
def read_arcgis_feature_layer(layer_url, batch_size=2000, out_sr=4326):
    """Download a public ArcGIS Feature Layer as a GeoDataFrame using its REST API."""
    query_url = layer_url.rstrip("/") + "/query"
    params = {
        "where": "1=1",
        "outFields": "*",
        "returnGeometry": "true",
        "outSR": out_sr,
        "f": "geojson",
        "resultRecordCount": batch_size,
    }

    features = []
    offset = 0
    while True:
        response = requests.get(
            query_url, params={**params, "resultOffset": offset}, timeout=60
        )
        response.raise_for_status()
        payload = response.json()

        if "error" in payload:
            raise RuntimeError(f"ArcGIS REST API error: {payload['error']}")

        batch = payload.get("features", [])
        features.extend(batch)
        if len(batch) < batch_size:
            break
        offset += len(batch)

    if not features:
        raise ValueError(f"No features returned from {layer_url}")

    return gpd.GeoDataFrame.from_features(features, crs=f"EPSG:{out_sr}")

point_df = read_arcgis_feature_layer(point_layer_url, out_sr=coordinate_system_id)
display(point_df.head())
print(f"Point features downloaded: {len(point_df):,}")

,geometry,OBJECTID,State_Name,County_Name,GeogType,ObservationDate,Abundance_Name,image1,Site_Name,Date_Updated,...,weedrank,InfestedAreaAcres,GrossAreaAcres,EDDS_ID,Subject_Na,Source,STATUS,ACRES,Project,Reporter
0,POINT (-105.04234 39.80979),1,Colorado,Jefferson,Point,1.307750e+12,Single Plant,NaN,Ralston Creek Trail,1485787923000,...,A,0.000247,NaN,1666083,"pale yellow iris, yellow flag iris",Point,1,1.0,3,john vickery
1,POINT (-105.00246 39.76064),2,Colorado,Denver,Point,1.307059e+12,Scattered Dense Patches,NaN,"City of Cuernavaca Park, E side of Platte River",1485788234000,...,A,0.000247,NaN,1666084,"pale yellow iris, yellow flag iris",Point,1,1.0,3,john vickery
2,POINT (-105.66129 40.41246),3,Colorado,Larimer,Point,1.311120e+12,Scattered Plants,NaN,Endovalley campground,1485787836000,...,A,1.000000,5.0,1960413,orange hawkweed,Point,1,1.0,1,Steve Priest
3,POINT (-105.13635 40.4124),4,Colorado,Larimer,Point,1.305072e+12,Scattered Dense Patches,NaN,Namaqua,1485787836000,...,A,50.000000,100.0,1960414,myrtle spurge,Point,1,50.0,1,Steve Priest
4,POINT (-105.29482 39.93268),5,Colorado,Boulder,Point,1.297555e+12,NaN,NaN,NaN,1488912279000,...,B,NaN,NaN,1998117,houndstongue,Point,1,1.0,15,Mike McHugh


Point features downloaded: 212,180


In [5]:
polygon_df = read_arcgis_feature_layer(polygon_layer_url, out_sr=coordinate_system_id)
display(polygon_df.head())
print(f"Polygon features downloaded: {len(polygon_df):,}")

,geometry,FID,Id,Shape__Area,Shape__Length
0,"POLYGON ((-105.01238 40.55843, -105.01248 40.5...",1,0,503713.418945,3615.652767


Polygon features downloaded: 1


## 4. Spatially join the two layers

GeoPandas performs the spatial join locally in the notebook. Each point is matched to the polygon it falls within. This does not publish a new hosted layer.

In [6]:
if point_df.crs != polygon_df.crs:
    point_df = point_df.to_crs(polygon_df.crs)

joined = gpd.sjoin(
    point_df, polygon_df, how="left", predicate="within",
    lsuffix="point", rsuffix="polygon"
)
display(joined.head())
print(f"Joined point records: {len(joined):,}")

,geometry,OBJECTID,State_Name,County_Name,GeogType,ObservationDate,Abundance_Name,image1,Site_Name,Date_Updated,...,Source,STATUS,ACRES,Project,Reporter,index_polygon,FID,Id,Shape__Area,Shape__Length
0,POINT (-105.04234 39.80979),1,Colorado,Jefferson,Point,1.307750e+12,Single Plant,NaN,Ralston Creek Trail,1485787923000,...,Point,1,1.0,3,john vickery,NaN,NaN,NaN,NaN,NaN
1,POINT (-105.00246 39.76064),2,Colorado,Denver,Point,1.307059e+12,Scattered Dense Patches,NaN,"City of Cuernavaca Park, E side of Platte River",1485788234000,...,Point,1,1.0,3,john vickery,NaN,NaN,NaN,NaN,NaN
2,POINT (-105.66129 40.41246),3,Colorado,Larimer,Point,1.311120e+12,Scattered Plants,NaN,Endovalley campground,1485787836000,...,Point,1,1.0,1,Steve Priest,NaN,NaN,NaN,NaN,NaN
3,POINT (-105.13635 40.4124),4,Colorado,Larimer,Point,1.305072e+12,Scattered Dense Patches,NaN,Namaqua,1485787836000,...,Point,1,50.0,1,Steve Priest,NaN,NaN,NaN,NaN,NaN
4,POINT (-105.29482 39.93268),5,Colorado,Boulder,Point,1.297555e+12,NaN,NaN,NaN,1488912279000,...,Point,1,1.0,15,Mike McHugh,NaN,NaN,NaN,NaN,NaN


Joined point records: 212,180


### Optional: count points in each polygon

This is the local equivalent of an aggregate-points workflow. It adds a `point_count` field to each polygon. It does not create or publish a hosted feature layer and does not consume ArcGIS credits.

In [7]:
result_layer_title = "Invasives_in_ELC"  # Descriptive label only; no hosted layer is created.

In [9]:

# Start with a copy of the polygons
polygon_counts = polygon_df.copy()

# Spatially join points to polygons.
# Each matching point receives the index of its polygon.
point_polygon_join = gpd.sjoin(
    point_df,
    polygon_df[["geometry"]],
    how="inner",
    predicate="within"
)

# Count points assigned to each polygon.
# The right-hand index column identifies the matching polygon.
counts = point_polygon_join.groupby("index_right").size()

# Match the counts to the original polygon index.
polygon_counts["point_count"] = (
    polygon_counts.index.map(counts)
    .fillna(0)
    .astype(int)
)

display(polygon_counts.drop(columns="geometry").head())

print(
    f"Polygons with at least one point: "
    f"{(polygon_counts['point_count'] > 0).sum():,}"
)


,FID,Id,Shape__Area,Shape__Length,point_count
0,1,0,503713.418945,3615.652767,148


Polygons with at least one point: 1


## 5. Visualize the spatial join result on a map

Folium displays an interactive map in the notebook. The map uses OpenStreetMap tiles, which require an internet connection.

In [10]:
polygons_map = polygon_counts.to_crs(epsg=4326)
points_map = point_df.to_crs(epsg=4326)

bounds = polygons_map.total_bounds  # minx, miny, maxx, maxy
center = [(bounds[1] + bounds[3]) / 2, (bounds[0] + bounds[2]) / 2]
map1 = folium.Map(location=center, zoom_start=12, tiles="OpenStreetMap")

folium.GeoJson(
    polygons_map,
    name="Polygons (point counts)",
    tooltip=folium.GeoJsonTooltip(fields=["point_count"], aliases=["Point count:"])
).add_to(map1)

# Plot at most 5,000 points to keep the map responsive.
points_to_map = points_map.head(5000)
folium.GeoJson(
    points_to_map,
    name="Points (first 5,000)",
    marker=folium.CircleMarker(radius=3, fill=True)
).add_to(map1)

folium.LayerControl().add_to(map1)
map1.fit_bounds([[bounds[1], bounds[0]], [bounds[3], bounds[2]]])
map1

## 6. Filter for points within the ELC

This section downloads the point and ELC polygon layers, then filters points locally using GeoPandas. It assumes the ELC layer contains the polygon(s) defining the target area.

In [11]:
invasives_url = point_layer_url
elc_url = polygon_layer_url

invasives_gdf = read_arcgis_feature_layer(invasives_url, out_sr=4326)
elc_gdf = read_arcgis_feature_layer(elc_url, out_sr=4326)

elc_geometry = elc_gdf.geometry.union_all()
invasives_in_elc = invasives_gdf[
    invasives_gdf.geometry.within(elc_geometry)
].copy()

print(f"Number of invasive points within ELC: {len(invasives_in_elc):,}")
display(invasives_in_elc.head())

Number of invasive points within ELC: 148


,geometry,OBJECTID,State_Name,County_Name,GeogType,ObservationDate,Abundance_Name,image1,Site_Name,Date_Updated,...,weedrank,InfestedAreaAcres,GrossAreaAcres,EDDS_ID,Subject_Na,Source,STATUS,ACRES,Project,Reporter
64167,POINT (-105.01856 40.55821),70546,Colorado,Larimer,Point,1.566950e+12,NaN,https://secure.bugwoodcloud.org/eddmaps/images...,NaN,1567026913000,...,B,NaN,NaN,10705981,common teasel,Point,1,1.0,86,iNaturalist Database
64168,POINT (-105.01842 40.55801),70547,Colorado,Larimer,Point,1.566950e+12,NaN,https://secure.bugwoodcloud.org/eddmaps/images...,NaN,1567026954000,...,B,NaN,NaN,10705982,Dalmatian toadflax,Point,1,1.0,86,iNaturalist Database
64250,POINT (-105.0185 40.55827),70634,Colorado,Larimer,Point,1.587600e+12,NaN,https://secure.bugwoodcloud.org/eddmaps/images...,NaN,1587667799000,...,B,NaN,NaN,10706069,common teasel,Point,1,1.0,86,iNaturalist Database
64545,POINT (-105.01818 40.55831),70959,Colorado,Larimer,Point,1.600560e+12,NaN,https://secure.bugwoodcloud.org/eddmaps/images...,NaN,1600624330000,...,B,NaN,NaN,10706396,Dalmatian toadflax,Point,1,1.0,86,iNaturalist Database
64546,POINT (-105.01816 40.55826),70960,Colorado,Larimer,Point,1.600560e+12,NaN,https://secure.bugwoodcloud.org/eddmaps/images...,NaN,1600624704000,...,B,NaN,NaN,10706397,Dalmatian toadflax,Point,1,1.0,86,iNaturalist Database


In [12]:
# Alternative spatial-join approach: retain points intersecting an ELC polygon.
invasives_within_elc = gpd.sjoin(
    invasives_gdf, elc_gdf, how="inner", predicate="intersects",
    lsuffix="point", rsuffix="elc"
)

print(f"Point records intersecting ELC: {len(invasives_within_elc):,}")
display(invasives_within_elc.head())

# Optional local exports:
# invasives_within_elc.to_file("Invasives_Within_ELC.geojson", driver="GeoJSON")
# invasives_within_elc.drop(columns="geometry").to_csv("Invasives_Within_ELC.csv", index=False)

Point records intersecting ELC: 148


,geometry,OBJECTID,State_Name,County_Name,GeogType,ObservationDate,Abundance_Name,image1,Site_Name,Date_Updated,...,Source,STATUS,ACRES,Project,Reporter,index_elc,FID,Id,Shape__Area,Shape__Length
64167,POINT (-105.01856 40.55821),70546,Colorado,Larimer,Point,1.566950e+12,NaN,https://secure.bugwoodcloud.org/eddmaps/images...,NaN,1567026913000,...,Point,1,1.0,86,iNaturalist Database,0,1,0,503713.418945,3615.652767
64168,POINT (-105.01842 40.55801),70547,Colorado,Larimer,Point,1.566950e+12,NaN,https://secure.bugwoodcloud.org/eddmaps/images...,NaN,1567026954000,...,Point,1,1.0,86,iNaturalist Database,0,1,0,503713.418945,3615.652767
64250,POINT (-105.0185 40.55827),70634,Colorado,Larimer,Point,1.587600e+12,NaN,https://secure.bugwoodcloud.org/eddmaps/images...,NaN,1587667799000,...,Point,1,1.0,86,iNaturalist Database,0,1,0,503713.418945,3615.652767
64545,POINT (-105.01818 40.55831),70959,Colorado,Larimer,Point,1.600560e+12,NaN,https://secure.bugwoodcloud.org/eddmaps/images...,NaN,1600624330000,...,Point,1,1.0,86,iNaturalist Database,0,1,0,503713.418945,3615.652767
64546,POINT (-105.01816 40.55826),70960,Colorado,Larimer,Point,1.600560e+12,NaN,https://secure.bugwoodcloud.org/eddmaps/images...,NaN,1600624704000,...,Point,1,1.0,86,iNaturalist Database,0,1,0,503713.418945,3615.652767
